# v5 비교 실험 — claude-haiku-4-5-20251001

- 같은 100명에게 `voter_context_v4` + `system_prompt_v4` 적용
- 페르소나 카드에 `[정치적 이해관계]` 블록 주입
- 모델: `claude-haiku-4-5-20251001` (Anthropic Claude Haiku 4.5, 스냅샷 고정)
- CSV 누적 시 `model = anthropic/claude-haiku-4-5-20251001`
- 결과: `vote_results_all.csv` 누적, raw `response/`

In [7]:
# [Cell 1] imports & env
import os, json, time, re
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

BASE = Path.cwd() if Path.cwd().name == "backend" else Path.cwd() / "backend"
CONTEXT_DIR = BASE / "context"
RESULTS_PATH = BASE / "vote_results_all.csv"
INTERESTS_PATH = BASE / "interests" / "personas_gpt-5.4-mini_all_interests.tsv"
RESPONSE_DIR = BASE / "response"
RESPONSE_DIR.mkdir(exist_ok=True)
LOG_DIR = BASE / "log"
LOG_DIR.mkdir(exist_ok=True)

MODEL = "claude-haiku-4-5-20251001"
PROVIDER = "anthropic"
VERSION = "v5"
MODEL_FULL = f"{PROVIDER}/{MODEL}"

# ANTHROPIC_API_KEY 는 backend/.env.local 또는 프로젝트 루트 .env.local 에서 로드
for env_path in [BASE / ".env.local", BASE.parent / ".env.local"]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"loaded: {env_path}")
        break
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY 필요"

def safe_filename(s: str) -> str:
    return re.sub(r"[^\w\-.]", "_", s)

print(f"model: {MODEL_FULL}, version: {VERSION}")

loaded: z:\Github\persona-million\.env.local
model: anthropic/claude-haiku-4-5-20251001, version: v5


In [8]:
# [Cell 2] gpt-5.4-mini × v2 페르소나 100명 로드 + political_interest 매핑 (페르소나는 gpt 결과에서 끌어옴)
df_all = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig")
mask = (df_all["model"] == "openai/gpt-5.4-mini") & (df_all["voter_context_version"] == "v2")
targets = df_all[mask].drop_duplicates(subset=["persona_uuid"]).reset_index(drop=True)
print(f"target personas: {len(targets)}")

interests = pd.read_csv(INTERESTS_PATH, sep="	", encoding="utf-8-sig")[["persona_uuid", "political_interest"]]
print(f"interests rows: {len(interests)}")

targets = targets.merge(interests, on="persona_uuid", how="left")
missing = targets["political_interest"].isna().sum()
print(f"political_interest 누락: {missing}/{len(targets)}")

# 이미 (현재 모델 × 현재 VERSION) 처리된 uuid 는 건너뛰기
done = set(df_all[(df_all["model"] == MODEL_FULL) & (df_all["voter_context_version"] == VERSION)]["persona_uuid"].tolist())
todo = targets[~targets["persona_uuid"].isin(done)].reset_index(drop=True)
print(f"이미 {MODEL_FULL} × {VERSION} 처리됨: {len(done)}, 남은 작업: {len(todo)}")


target personas: 100
interests rows: 100
political_interest 누락: 0/100
이미 anthropic/claude-haiku-4-5-20251001 × v5 처리됨: 0, 남은 작업: 100


In [9]:
# [Cell 3] v4 프롬프트 + Anthropic 체인 + persona_card with political_interest
def load_context(version: str) -> str:
    return (CONTEXT_DIR / f"voter_context_{version}.md").read_text(encoding="utf-8")
def load_system_prompt(version: str) -> str:
    return (CONTEXT_DIR / f"system_prompt_{version}.md").read_text(encoding="utf-8")

POLITICAL_CONTEXT = load_context(VERSION)
SYSTEM_TEMPLATE = load_system_prompt(VERSION)
print(f"voter_context_{VERSION}.md: {len(POLITICAL_CONTEXT)} chars")
print(f"system_prompt_{VERSION}.md: {len(SYSTEM_TEMPLATE)} chars")

USER_TEMPLATE = """다음 페르소나가 2026년 6월 3일 지방선거(또는 가까운 미래의 총선·대선)에서 어느 정당을 지지·투표할지 추론하시오.

{persona_card}

JSON만 출력."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("user", USER_TEMPLATE),
])

# Anthropic 은 OpenAI 의 response_format JSON mode 같은 게 없음 → 시스템 프롬프트의 JSON 강제 규칙에 의존
llm = ChatAnthropic(
    model=MODEL,
    temperature=0.7,
    max_tokens=1500,
)
chain = prompt | llm

VOTE_KEYS = ("vote", "party", "정당", "지지정당", "투표", "선택", "지지")
REASON_KEYS = ("reason", "이유", "rationale", "사유", "근거", "설명")

def _try_complete_json(s: str) -> str:
    last_close = s.rfind("}")
    last_open = s.rfind("{")
    if last_open == -1: return s
    if last_close > last_open: return s[last_open:last_close + 1]
    candidate = s[last_open:].rstrip().rstrip(",")
    if candidate.count('"') % 2 == 1: candidate += '"'
    candidate += "}"
    return candidate

def parse_response(raw: str) -> dict:
    s = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    s = re.sub(r"```(?:json)?\s*", "", s).replace("```", "")
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    try: return json.loads(_try_complete_json(s))
    except json.JSONDecodeError:
        raise ValueError(f"no parseable JSON: {raw[:200]!r}")

def pick(d, keys):
    for k in keys:
        v = d.get(k)
        if v: return v
    return None

def persona_card(r) -> str:
    fields = [
        ("성별", r["sex"]), ("연령", f"{r['age']}세"),
        ("혼인상태", r["marital_status"]),
        ("가구형태", r["family_type"]), ("주거", r["housing_type"]),
        ("학력", r["education_level"]), ("전공", r["bachelors_field"]),
        ("직업", r["occupation"]),
        ("지역", f"{r['province']} {r['district']}"),
    ]
    demo = "\n".join(f"- {k}: {v}" for k, v in fields if v)
    narr = (
        f"[요약]\n{r['persona_summary']}\n\n"
        f"[직업적 면모]\n{r['professional_persona']}\n\n"
        f"[가족 면모]\n{r['family_persona']}\n\n"
        f"[문화적 배경]\n{r['cultural_background']}\n\n"
        f"[관심사]\n{r['hobbies_and_interests']}\n\n"
        f"[목표]\n{r['career_goals_and_ambitions']}"
    )
    out = f"## 인구통계\n{demo}\n\n## 인물 서사\n{narr}"
    pi = r.get("political_interest")
    if isinstance(pi, str) and pi.strip():
        out += f"\n\n## 정치적 이해관계\n[정치적 이해관계]\n{pi.strip()}"
    return out

if len(todo) > 0:
    print("\n--- 첫 페르소나 카드 미리보기 ---")
    print(persona_card(todo.iloc[0])[:1200])

voter_context_v5.md: 5475 chars
system_prompt_v5.md: 1778 chars

--- 첫 페르소나 카드 미리보기 ---
## 인구통계
- 성별: 여자
- 연령: 79세
- 혼인상태: 배우자있음
- 가구형태: 배우자와 거주
- 주거: 단독주택
- 학력: 초등학교
- 전공: 해당없음
- 직업: 건물 청소원
- 지역: 충청남 충청남-금산군

## 인물 서사
[요약]
장말순 씨는 금산 보건소의 보이지 않는 곳을 묵묵히 가꾸며, 소박한 노동과 정겨운 이웃 관계 속에서 삶의 안정을 찾는 79세의 강단 있는 할머니입니다.

[직업적 면모]
장말순 씨는 금산군 보건소 복도의 찌든 때를 없애기 위해 락스와 세제를 자신만의 비율로 섞어 사용하며, 내원객들이 붐비는 시간대를 피해 소리 없이 빠르게 바닥을 닦아내는 능숙한 청소원입니다.

[가족 면모]
장말순 씨는 금산의 낡은 단독주택에서 무뚝뚝하지만 평생 곁을 지켜준 남편과 함께 살며, 서로의 존재만으로도 충분한 고요하고 안정적인 노후를 보내고 있습니다.

[문화적 배경]
충남 금산의 인삼 밭이 내려다보이는 마을에서 나고 자라며 흙과 함께 유년 시절을 보냈습니다. 정규 교육은 짧았지만, 마을 공동체 안에서 서로 돕고 나누는 정서와 금산 특유의 느긋하면서도 강단 있는 생활 방식을 몸에 익혔습니다.

[관심사]
금강변 산책로를 천천히 걸으며 계절마다 변하는 풍경을 구경하거나, 동네 친구들과 함께 읍내 목욕탕에서 뜨끈한 물에 몸을 지지며 밀린 이야기를 나누는 시간을 보냅니다.

[목표]
몸이 허락하는 한 지금 하는 청소 일을 계속 유지하며, 매달 받는 월급으로 남편과 함께 가끔 외식을 즐기고 소소한 생활비를 보태는 안정적인 일상을 유지하고자 합니다.

## 정치적 이해관계
[정치적 이해관계]
올해 일흔아홉, 금산군 보건소에서 "락스와 세제를 자신만의 비율로 섞어" 복도를 닦고 있다. 새벽에 라디오를 틀면 2026년 기초연금이 단독가구 월 40만 원까지 오른다, 선정기준액이 247만 원으로 높아졌다는

In [10]:
# [Cell 3.5] 1명만 호출해서 응답·파싱·시간 확인 (스모크 테스트)
if len(todo) > 0:
    r = todo.iloc[0]
    t0 = time.perf_counter()
    msg = chain.invoke({"political_context": POLITICAL_CONTEXT, "persona_card": persona_card(r)})
    el = time.perf_counter() - t0
    raw = msg.content if hasattr(msg, "content") else str(msg)
    print(f"[elapsed] {el:.2f}s\n")
    print("--- raw ---")
    print(raw)
    print("\n--- parsed ---")
    p = parse_response(raw)
    print("vote:", pick(p, VOTE_KEYS))
    print("reason:", pick(p, REASON_KEYS))

[elapsed] 4.93s

--- raw ---
```json
{"vote": "더불어민주당", "reason": "기초연금이 월 40만 원까지 올라가고 선정기준액이 247만 원으로 높아진 것이 직접적으로 우리 생활을 지탱하는 일입니다. 영감 약값, 달마다 먹는 단팥빵 값—이 모든 게 그 돈에서 나오니까요. 보건소 복도를 닦으며 본 현실은 더 심각합니다. 내가 일하는 진료실 옆이 자꾸 비어가고, 금산 같은 곳에서 의사가 없어지면 노인들은 어디로 가나 싶어요. 대전까지 버스 타고 가야 하는데 농어촌버스도 줄어드는 형편입니다. 이재명 정부가 노인일자리 115만 개를 만들고 기초연금을 올린 것이 우리 같은 농촌 노인들을 생각한 정책 같습니다. 최저임금도 올라가서 청소 일을 계속 할 수 있으면 좋겠고요. 물론 그 정책이 용역 일자리에 제대로 반영될지는 걱정이지만, 적어도 농촌 의료와 노인 복지를 챙기려는 정부가 우리 같은 사람들을 보고 있다는 느낌이 듭니다. 영감이랑 한 달에 한 번 삼겹살집 가는 낙을 지킬 수 있어야 하니까요."}
```

--- parsed ---
vote: 더불어민주당
reason: 기초연금이 월 40만 원까지 올라가고 선정기준액이 247만 원으로 높아진 것이 직접적으로 우리 생활을 지탱하는 일입니다. 영감 약값, 달마다 먹는 단팥빵 값—이 모든 게 그 돈에서 나오니까요. 보건소 복도를 닦으며 본 현실은 더 심각합니다. 내가 일하는 진료실 옆이 자꾸 비어가고, 금산 같은 곳에서 의사가 없어지면 노인들은 어디로 가나 싶어요. 대전까지 버스 타고 가야 하는데 농어촌버스도 줄어드는 형편입니다. 이재명 정부가 노인일자리 115만 개를 만들고 기초연금을 올린 것이 우리 같은 농촌 노인들을 생각한 정책 같습니다. 최저임금도 올라가서 청소 일을 계속 할 수 있으면 좋겠고요. 물론 그 정책이 용역 일자리에 제대로 반영될지는 걱정이지만, 적어도 농촌 의료와 노인 복지를 챙기려는 정부가 우리 같은 사람들을 보고 있다는 느낌이 듭니다. 영감이랑 한 달에 한 번 

In [11]:
# [Cell 4] 100명 v4 추론 (Haiku) → CSV 누적 + response/ 저장
log_path = LOG_DIR / f"{time.strftime('%Y%m%d-%H%M%S')}_voteV5_anthropic_{safe_filename(MODEL)}_{VERSION}.log"
log_f = log_path.open("w", encoding="utf-8")
def log(msg):
    print(msg, flush=True)
    log_f.write(msg + "\n"); log_f.flush()

log(f"=== {MODEL_FULL} × {len(todo)} (version={VERSION}) ===")
success = 0
for i, r in todo.iterrows():
    t0 = time.perf_counter()
    raw = ""
    try:
        msg = chain.invoke({"political_context": POLITICAL_CONTEXT, "persona_card": persona_card(r)})
        raw = msg.content if hasattr(msg, "content") else str(msg)
        p = parse_response(raw)
        vote = pick(p, VOTE_KEYS)
        reason = pick(p, REASON_KEYS)
        if not vote or not reason:
            raise ValueError(f"missing keys: {list(p.keys())}")
    except Exception as e:
        el = time.perf_counter() - t0
        log(f"[{i+1:>3}/{len(todo)}] ERROR ({el:.1f}s): {str(e)[:120]}")
        ts = time.strftime("%Y%m%d-%H%M%S")
        fname = f"FAIL_{ts}_anthropic_{safe_filename(MODEL)}_{r['persona_uuid'][:8]}_{VERSION}.txt"
        (RESPONSE_DIR / fname).write_text(
            f"=== ERROR: {e}\n=== MODEL: {MODEL_FULL}\n=== UUID: {r['persona_uuid']}\n\n{raw}",
            encoding="utf-8")
        continue
    el = time.perf_counter() - t0
    success += 1

    ts = time.strftime("%Y%m%d-%H%M%S")
    fname = f"{ts}_anthropic_{safe_filename(MODEL)}_{r['persona_uuid'][:8]}_{VERSION}_{VERSION}.json"
    out = {
        "timestamp": ts,
        "model": MODEL_FULL,
        "voter_context_version": VERSION,
        "system_prompt_version": VERSION,
        "persona_uuid": r["persona_uuid"],
        "elapsed_sec": round(el, 3),
        "persona": {
            "sex": r["sex"], "age": int(r["age"]),
            "marital_status": r["marital_status"],
            "family_type": r["family_type"],
            "housing_type": r["housing_type"],
            "education_level": r["education_level"],
            "bachelors_field": r["bachelors_field"],
            "occupation": r["occupation"],
            "province": r["province"], "district": r["district"],
            "persona_summary": r["persona_summary"],
        },
        "political_interest": r.get("political_interest"),
        "raw_response": raw,
        "parsed": {"vote": vote, "reason": reason},
    }
    (RESPONSE_DIR / fname).write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

    rec = {
        "persona_uuid": r["persona_uuid"],
        "sex": r["sex"], "age": int(r["age"]),
        "marital_status": r["marital_status"],
        "family_type": r["family_type"], "housing_type": r["housing_type"],
        "education_level": r["education_level"],
        "bachelors_field": r["bachelors_field"],
        "occupation": r["occupation"],
        "province": r["province"], "district": r["district"],
        "persona_summary": r["persona_summary"],
        "vote": vote, "reason": reason,
        "model": MODEL_FULL,
        "elapsed_sec": round(el, 3),
        "response_file": fname,
        "voter_context_version": VERSION,
        "system_prompt_version": VERSION,
        "professional_persona": r["professional_persona"],
        "sports_persona": r["sports_persona"],
        "arts_persona": r["arts_persona"],
        "travel_persona": r["travel_persona"],
        "culinary_persona": r["culinary_persona"],
        "family_persona": r["family_persona"],
        "cultural_background": r["cultural_background"],
        "skills_and_expertise": r["skills_and_expertise"],
        "hobbies_and_interests": r["hobbies_and_interests"],
        "career_goals_and_ambitions": r["career_goals_and_ambitions"],
    }
    df_rec = pd.DataFrame([rec])
    if RESULTS_PATH.exists():
        df_rec.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_rec.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
    log(f"[{i+1:>3}/{len(todo)}] {r['sex']} {r['age']}세 {r['province']:<6} "
        f"{str(r['occupation'])[:14]:<14} → {vote[:10]:<10} ({el:.1f}s)")

log(f"\n결과: {success}/{len(todo)} 성공")
log_f.close()

=== anthropic/claude-haiku-4-5-20251001 × 100 (version=v5) ===
[  1/100] 여자 79세 충청남    건물 청소원         → 더불어민주당     (5.8s)
[  2/100] 남자 29세 대구     무직             → 더불어민주당     (6.1s)
[  3/100] 남자 27세 서울     전직 금속제품 생산 관리자 → 개혁신당       (6.4s)
[  4/100] 여자 71세 대전     무직             → 더불어민주당     (5.0s)
[  5/100] 여자 52세 부산     전동차 정비원        → 더불어민주당     (6.3s)
[  6/100] 여자 20세 서울     무직             → 무당층/기권     (4.6s)
[  7/100] 여자 26세 제주     초등학교 교사        → 더불어민주당     (4.5s)
[  8/100] 남자 45세 경상남    용접기 조작원        → 더불어민주당     (5.7s)
[  9/100] 여자 63세 충청북    요양 보호사         → 더불어민주당     (5.2s)
[ 10/100] 여자 67세 경기     무직             → 더불어민주당     (4.2s)
[ 11/100] 여자 40세 강원     혼례 종사원         → 무당층/기권     (5.2s)
[ 12/100] 남자 48세 서울     마케팅 전문가        → 개혁신당       (15.3s)
[ 13/100] 남자 45세 인천     그 외 택배원        → 더불어민주당     (6.0s)
[ 14/100] 남자 38세 충청남    한식 조리사         → 더불어민주당     (14.8s)
[ 15/100] 여자 26세 서울     보육교사           → 더불어민주당     (11.6s)
[ 16/100] 남자 53세 부산     대형 화물차 운전원     → 국민의힘    

In [12]:
# [Cell 5] 모델·버전 분포 비교 (gpt v2/v3/v4/v5 + haiku v4/v5)
df_all = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig")
combos = [
    ("openai/gpt-5.4-mini", "v2"),
    ("openai/gpt-5.4-mini", "v3"),
    ("openai/gpt-5.4-mini", "v4"),
    ("openai/gpt-5.4-mini", "v5"),
    (MODEL_FULL, "v4"),
    (MODEL_FULL, "v5"),
]
for m, v in combos:
    sub = df_all[(df_all["model"] == m) & (df_all["voter_context_version"] == v)]
    sub = sub.drop_duplicates("persona_uuid", keep="last")
    if len(sub) == 0:
        continue
    print(f"=== {m} / {v} (n={len(sub)}) ===")
    print(sub["vote"].value_counts())
    print()


=== openai/gpt-5.4-mini / v2 (n=100) ===
vote
국민의힘      42
더불어민주당    40
무당층/기권    10
개혁신당       8
Name: count, dtype: int64

=== openai/gpt-5.4-mini / v3 (n=100) ===
vote
더불어민주당    49
국민의힘      37
개혁신당       9
무당층/기권     5
Name: count, dtype: int64

=== openai/gpt-5.4-mini / v4 (n=100) ===
vote
더불어민주당    79
국민의힘      11
개혁신당       8
진보당        1
무당층/기권     1
Name: count, dtype: int64

=== openai/gpt-5.4-mini / v5 (n=100) ===
vote
더불어민주당    93
국민의힘       6
진보당        1
Name: count, dtype: int64

=== anthropic/claude-haiku-4-5-20251001 / v4 (n=100) ===
vote
더불어민주당    50
무당층/기권    41
국민의힘       9
Name: count, dtype: int64

=== anthropic/claude-haiku-4-5-20251001 / v5 (n=99) ===
vote
더불어민주당    74
무당층/기권    12
국민의힘       8
개혁신당       5
Name: count, dtype: int64

